# Tea Shop Sales Descriptive & Predictive Analytics
**Practical Report — Historical Data Summarization, Visual Profiling & Trend Forecasting**

* **Period:** October 1 – October 10, 2026
* **Tools:** Python (`pandas`, `numpy`, `matplotlib`)

## 1. Import Libraries & Load Dataset
We create the 20-row dataset covering 10 consecutive days of tea shop sales across four products: **Masala Chai**, **Ginger Tea**, **Green Tea**, and **Iced Peach Tea**.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Create dataset (20 rows: 10 days, 2 products sold per day)
data = {
    "Date": [
        "2026-10-01", "2026-10-01", "2026-10-02", "2026-10-02", "2026-10-03",
        "2026-10-03", "2026-10-04", "2026-10-04", "2026-10-05", "2026-10-05",
        "2026-10-06", "2026-10-06", "2026-10-07", "2026-10-07", "2026-10-08",
        "2026-10-08", "2026-10-09", "2026-10-09", "2026-10-10", "2026-10-10"
    ],
    "Product": [
        "Masala Chai", "Ginger Tea", "Green Tea", "Masala Chai", "Iced Peach Tea",
        "Ginger Tea", "Masala Chai", "Green Tea", "Iced Peach Tea", "Masala Chai",
        "Ginger Tea", "Iced Peach Tea", "Green Tea", "Masala Chai", "Ginger Tea",
        "Iced Peach Tea", "Masala Chai", "Green Tea", "Iced Peach Tea", "Ginger Tea"
    ],
    "Cups_Sold": [
        30, 25, 20, 28, 35, 22, 40, 18, 38, 32,
        27, 42, 21, 35, 30, 45, 38, 24, 50, 33
    ],
    "Price": [
        180, 160, 120, 180, 200, 160, 180, 120, 200, 180,
        160, 200, 120, 180, 160, 200, 180, 120, 200, 160
    ]
}

df = pd.DataFrame(data)
df["Revenue"] = df["Cups_Sold"] * df["Price"]

# Save dataset to CSV
df.to_csv("tea_sales.csv", index=False)
df.head(10)

## 2. Descriptive Analytics — Core Business Questions
* **Q1:** What was the total gross revenue and volume of cups sold over the 10 days?
* **Q2:** How do individual products compare in total volume, gross revenue, and share of sales?
* **Q3:** What are the daily sales figures, highest day, lowest day, and daily average?
* **Q4:** What are the central tendencies (mean, median, standard deviation) for cups sold per product?

In [ ]:
# Q1: Overall Sales Metrics
total_revenue = df["Revenue"].sum()
total_cups = df["Cups_Sold"].sum()
avg_rev_per_cup = total_revenue / total_cups

print("=== Q1: OVERALL SALES METRICS ===")
print(f"Total Gross Revenue : Rs {total_revenue:,}")
print(f"Total Cups Sold     : {total_cups}")
print(f"Avg Revenue / Cup   : Rs {avg_rev_per_cup:.2f}")

# Q2: Category / Product Performance Breakdown
product_summary = df.groupby("Product").agg(
    Total_Cups=("Cups_Sold", "sum"),
    Total_Revenue=("Revenue", "sum")
).sort_values("Total_Revenue", ascending=False)

product_summary["Volume_Share_%"] = (product_summary["Total_Cups"] / total_cups * 100).round(1)
product_summary["Revenue_Share_%"] = (product_summary["Total_Revenue"] / total_revenue * 100).round(1)

print("\n=== Q2: PRODUCT PERFORMANCE BREAKDOWN ===")
print(product_summary)
print(f"Best Performing Product: {product_summary.index[0]}")

# Q3: Daily Revenue & Variance Analysis
daily_summary = df.groupby("Date").agg(
    Total_Cups=("Cups_Sold", "sum"),
    Daily_Revenue=("Revenue", "sum")
)
avg_daily_rev = daily_summary["Daily_Revenue"].mean()
daily_summary["Vs_Average_%"] = ((daily_summary["Daily_Revenue"] - avg_daily_rev) / avg_daily_rev * 100).round(1)

print("\n=== Q3: DAILY REVENUE BREAKDOWN ===")
print(daily_summary)
print(f"Average Daily Revenue: Rs {avg_daily_rev:,.2f}")
print(f"Peak Day             : {daily_summary['Daily_Revenue'].idxmax()} (Rs {daily_summary['Daily_Revenue'].max():,})")
print(f"Lowest Day           : {daily_summary['Daily_Revenue'].idxmin()} (Rs {daily_summary['Daily_Revenue'].min():,})")

# Q4: Statistical Distribution (Mean, Median, Std Dev)
stats_dist = df.groupby("Product")["Cups_Sold"].agg(
    Mean="mean",
    Median="median",
    Std_Dev="std"
).round(2)

print("\n=== Q4: STATISTICAL DISTRIBUTION OF CUPS SOLD ===")
print(stats_dist)

## 3. Data Visualizations
Visualizing product revenue (bar chart), daily revenue trend (line chart), and share of cups sold (pie chart).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Graph 1: Revenue by Product (Bar Chart)
product_summary["Total_Revenue"].plot(kind="bar", color="#2E6F40", ax=axes[0])
axes[0].set_title("Revenue by Tea Product", fontsize=12, fontweight="bold")
axes[0].set_ylabel("Revenue (Rs)")
axes[0].set_xlabel("Product")
axes[0].tick_params(axis="x", rotation=15)

# Graph 2: Daily Revenue Trend (Line Chart)
daily_summary["Daily_Revenue"].plot(kind="line", marker="o", color="#D97706", linewidth=2, ax=axes[1])
axes[1].axhline(avg_daily_rev, color="gray", linestyle="--", label=f"Avg: Rs {avg_daily_rev:,.0f}")
axes[1].set_title("Daily Revenue Trend (Oct 1 - Oct 10)", fontsize=12, fontweight="bold")
axes[1].set_ylabel("Revenue (Rs)")
axes[1].set_xlabel("Date")
axes[1].tick_params(axis="x", rotation=45)
axes[1].legend()

# Graph 3: Share of Cups Sold (Pie Chart)
product_summary["Total_Cups"].plot(
    kind="pie",
    autopct="%1.1f%%",
    colors=["#2E6F40", "#689F38", "#D97706", "#A3E635"],
    ax=axes[2]
)
axes[2].set_title("Share of Cups Sold", fontsize=12, fontweight="bold")
axes[2].set_ylabel("")

plt.tight_layout()
plt.show()

## 4. Predictive Analytics — Linear Trend Forecasting (Day 11)
Using Ordinary Least Squares (OLS) linear regression (`numpy.polyfit`) to model the daily revenue trend and predict Day 11 (`2026-10-11`) revenue.

In [ ]:
# Day index 1 to 10
days = np.arange(1, len(daily_summary) + 1)
revenues = daily_summary["Daily_Revenue"].values

# Fit linear trend line: Revenue = m * Day + c
slope, intercept = np.polyfit(days, revenues, 1)
predicted_day_11 = slope * 11 + intercept

print("=== PREDICTIVE FORECASTING (OLS LINEAR TREND) ===")
print(f"Trend Equation           : Revenue = {slope:.2f} * Day + {intercept:.2f}")
print(f"Daily Growth Rate        : +Rs {slope:.2f} per day")
print(f"Forecasted Day 11 Revenue: Rs {predicted_day_11:,.2f}")

# Plot historical vs predicted trend
plt.figure(figsize=(9, 4.5))
plt.plot(days, revenues, marker="o", label="Actual Daily Revenue", color="#2E6F40")
plt.plot(np.append(days, 11), slope * np.append(days, 11) + intercept, linestyle="--", color="#D97706", label="Linear Trend Line")
plt.scatter([11], [predicted_day_11], color="red", zorder=5, s=80, label=f"Day 11 Forecast (Rs {predicted_day_11:,.0f})")
plt.title("Daily Revenue Forecast (Linear Trend Model)", fontweight="bold")
plt.xlabel("Day Number (Oct 1 = Day 1)")
plt.ylabel("Revenue (Rs)")
plt.xticks(range(1, 12))
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()